In [1]:
#data ingestion 
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("E:/YT/New folder/1.docx")
text_document = loader.load()

content = text_document[0].page_content

C:\Users\kanan\AppData\Local\Temp\ipykernel_30600\2414634538.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"


In [3]:
#web based loader
from langchain_community.document_loaders import PlaywrightURLLoader
import bs4
load_dotenv()
os.environ["USER_AGENT"] = os.getenv("USER_AGENT")

"""loader = WebBaseLoader(web_path=("https://medium.com/@akshat.g_77864/free-and-paid-large-language-models-with-langchain-5950033b8c7d",),
                       bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                           class_=("flex-1 min-w-0 space-y-8")
                       )),)"""

loader = PlaywrightURLLoader(
    urls=['https://medium.com/@akshat.g_77864/free-and-paid-large-language-models-with-langchain-5950033b8c7d'],
    remove_selectors=['header','footer','script','style']
)

web_page = loader.load()



Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

In [3]:
# web based loader
from langchain_community.document_loaders import WebBaseLoader
import bs4

## load,chunk and index the content of the html page

loader=WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
                     bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                         class_=("post-title","post-content","post-header")

                     )))

web_page=loader.load()

In [9]:
#split docs to chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=300)
doc = text_splitter.split_documents(web_page)

In [10]:
len(doc),doc[:4]

(63,
 [Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes an

In [11]:
#chunks to vector using vector embedding and vector store
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

db = Chroma.from_documents(doc[:10],embedding)

In [12]:
type(db)

langchain_community.vectorstores.chroma.Chroma

In [15]:
query = "what is this doc about"
results = db.similarity_search(query)
results

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI,'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Agent System Overview#'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='and programs; it can be framed as a powerful general problem solver.'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng')]

In [14]:
results[1].page_content

'Agent System Overview#'

In [17]:
##Faiss vectordb

from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(doc[:10],embedding)
query = "what is this doc about"
results = faiss_db.similarity_search(query)
results

[Document(id='224f2881-1ad8-43b5-bb8e-560b8e0018ed', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-f